# 001_nb_utils
Shared functions for every loader and orchestrator: control-table lookups, run logging,
table writes, script execution and the dependency-wave engine.

Needs `000_nb_configurations` loaded first (the bootstrap cell of each notebook does both).

In [ ]:
import time
import uuid
import threading
import traceback
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql import functions as F
from pyspark.sql import types as T

## Control tables
`ctrl.data_flow_control` holds one row per flow, `ctrl.source_type_registry` maps each
`source_type` to the loader notebook for its layer.

In [ ]:
def get_flow_config(control_id):
    rows = spark.sql(f"SELECT * FROM {CTRL_DATABASE}.data_flow_control WHERE control_id = {int(control_id)}").collect()
    if not rows:
        raise ValueError(f"control_id {control_id} is not in {CTRL_DATABASE}.data_flow_control")
    return rows[0].asDict()


def get_active_flows(layer):
    df = spark.sql(f"""
        SELECT *
        FROM {CTRL_DATABASE}.data_flow_control
        WHERE is_active = true
          AND layer = '{layer.upper()}'
        ORDER BY load_sequence, control_id
    """)
    return [row.asDict() for row in df.collect()]


def get_registry(layer):
    rows = spark.sql(f"""
        SELECT source_type, notebook_name
        FROM {CTRL_DATABASE}.source_type_registry
        WHERE is_active = true
          AND layer = '{layer.upper()}'
    """).collect()
    return {row["source_type"]: row["notebook_name"] for row in rows}


def dependencies_of(flow):
    raw = flow.get("depends_on_control_ids") or ""
    return [int(part) for part in raw.split(",") if part.strip()]

## Run log
Append-only: one `RUNNING` row when a flow starts and one final row when it ends, both with the
same `run_id`. The Databricks version updates the row in place; locally, an append-only log with
one write at a time is the simplest way to stay safe while flows run in parallel. `052_pipeline_run_report` shows the
latest row per run.

In [ ]:
RUN_LOG_SCHEMA = T.StructType([
    T.StructField("run_id", T.StringType(), False),
    T.StructField("control_id", T.LongType(), False),
    T.StructField("batch_id", T.StringType(), True),
    T.StructField("pipeline_name", T.StringType(), False),
    T.StructField("layer", T.StringType(), False),
    T.StructField("status", T.StringType(), False),
    T.StructField("retry_attempt", T.IntegerType(), False),
    T.StructField("event_time", T.TimestampType(), False),
    T.StructField("duration_seconds", T.DoubleType(), True),
    T.StructField("source_object", T.StringType(), True),
    T.StructField("destination_table", T.StringType(), True),
    T.StructField("rows_written", T.LongType(), True),
    T.StructField("error_message", T.StringType(), True),
    T.StructField("triggered_by", T.StringType(), False),
])


RUN_LOG_LOCK = threading.Lock()


def literal_frame(values, schema):
    """A DataFrame built inside Spark from Python values: one row, or none if values is None.

    spark.createDataFrame() pickles Python rows into a Python worker, which breaks on Python
    versions newer than Spark supports. Literal columns stay inside the JVM, so this works on
    any Python.
    """
    row_count = 0 if values is None else 1
    values = values or {}
    return spark.range(row_count).select([
        F.lit(values.get(field.name)).cast(field.dataType).alias(field.name)
        for field in schema.fields
    ])


def append_run_event(**event):
    event["event_time"] = datetime.now()
    # Flows run in parallel threads; one log write at a time keeps the log table consistent.
    with RUN_LOG_LOCK:
        (
            literal_frame(event, RUN_LOG_SCHEMA)
            .write.format(TABLE_FORMAT)
            .mode("append")
            .save(ctrl_table_path("pipeline_run_log").as_posix())
        )


def new_batch_id():
    return f"batch_{datetime.now().strftime('%Y%m%d_%H%M%S')}"


def latest_status(control_id):
    """Status of the most recent finished run of a flow (any batch), or None."""
    spark.catalog.refreshTable(f"{CTRL_DATABASE}.pipeline_run_log")  # see appends from this session
    rows = spark.sql(f"""
        SELECT status
        FROM {CTRL_DATABASE}.pipeline_run_log
        WHERE control_id = {int(control_id)}
          AND status <> 'RUNNING'
        ORDER BY event_time DESC
        LIMIT 1
    """).collect()
    return rows[0]["status"] if rows else None

## Writing tables
Every layer write is a full overwrite of a Delta table (atomic, with the previous version kept
for time travel), followed by registering the table name.

In [ ]:
import shutil


def reregister_table(layer, business_schema, table_name):
    database = layer_database(layer, business_schema)
    if TABLE_FORMAT != "delta":
        # A plain parquet table keeps the schema it was registered with, so register it again
        spark.sql(f"DROP TABLE IF EXISTS {database}.{table_name}")
    register_table(database, table_name, table_path(layer, business_schema, table_name))
    spark.catalog.refreshTable(f"{database}.{table_name}")


def write_table(df, layer, business_schema, table_name):
    path = table_path(layer, business_schema, table_name)
    writer = df.write.format(TABLE_FORMAT).mode("overwrite")
    if TABLE_FORMAT == "delta":
        writer = writer.option("overwriteSchema", "true")
    writer.save(path.as_posix())
    reregister_table(layer, business_schema, table_name)
    return spark.table(table_full_name(layer, business_schema, table_name)).count()


def with_lineage(df, batch_id):
    """Every Silver and Gold row records the batch that last wrote it. Incremental flows use it
    as their watermark. The publish step leaves columns starting with _ out of the Power BI files."""
    return df.withColumn("_batch_id", F.lit(batch_id))

## Incremental loads
Production pattern: after the first load, a flow only processes what is new.

| load_type | Used for | How |
|---|---|---|
| `SNAPSHOT` | Bronze reference files | Replace the table with the latest snapshot, append it to `<table>_snapshots` |
| `INCREMENTAL` | Bronze and Silver ledger and till feed | Watermark, then append (Bronze) or MERGE on the key (Silver) |
| `FULL` | Silver reference tables, budget, checks | Replace the table (type 1: the latest values win) |
| `SCD2` | Silver `store_history` | Compare the latest snapshot with the current versions: close changed and removed members, open new versions |
| `MV_FULL` | Gold materialized views (dimensions, small facts) | Recompute from the definition |
| `MV_INCREMENTAL` | Gold materialized views (large facts) | Recompute only the periods touched by new Silver rows (`replaceWhere`) |

History, change capture and incremental loading live in Bronze and Silver. Gold holds only
materialized views: a SQL definition plus a stored result, refreshed from Silver.

The watermark is read from the target itself (its highest `watermark_column`), so a failed or
skipped run is picked up automatically by the next one. A flow loads in full on its first run,
or when the orchestrator is run with `full_refresh=True` (needed after a change to reference
data such as the mapping workbook or FX rates, because older rows were built with the old values).
SCD2 history is never rebuilt by a full refresh.

The writes use Delta `MERGE` and `replaceWhere`. In parquet test mode the same result is written
through a staging folder instead.

In [ ]:
def table_exists(layer, business_schema, table_name):
    return is_table_folder(table_path(layer, business_schema, table_name))


def write_snapshot(df, layer, business_schema, table_name):
    """Reference files arrive as full snapshots. The table holds the latest snapshot, and
    <table>_snapshots keeps every delivery (append-only, by _batch_id), so the history of the
    source is never lost and any past state can be replayed."""
    rows = write_table(df, layer, business_schema, table_name)
    snapshots = f"{table_name}_snapshots"
    if table_exists(layer, business_schema, snapshots):
        df.write.format(TABLE_FORMAT).mode("append").save(table_path(layer, business_schema, snapshots).as_posix())
        reregister_table(layer, business_schema, snapshots)
    else:
        write_table(df, layer, business_schema, snapshots)
    return rows


def read_fresh(full_name):
    spark.catalog.refreshTable(full_name)
    return spark.table(full_name)


def max_value(full_name, column):
    return read_fresh(full_name).agg(F.max(column).alias("high")).collect()[0]["high"]


def sql_literal(value):
    if isinstance(value, (int, float)):
        return str(value)
    if hasattr(value, "isoformat"):
        return f"DATE '{value.isoformat()}'"
    return "'" + str(value).replace("'", "''") + "'"


def resolve_reference(reference, business_schema):
    """'silver.gl_lines' -> silver_wahat.gl_lines"""
    layer, table_name = reference.split(".")
    return table_full_name(layer, business_schema, table_name)


def prepare_increment(config):
    """Rows of the flow's incremental_source that its target has not processed yet, registered
    as a temp view that the flow's script reads as {{increment}}. Returns (rows, is_full_load)."""
    schema = config["destination_schema"]
    target_layer = config["layer"].lower()
    target = table_full_name(target_layer, schema, config["destination_table"])
    watermark = config["watermark_column"]
    source = read_fresh(resolve_reference(config["incremental_source"], schema))

    is_full_load = bool(config.get("full_refresh")) or not table_exists(target_layer, schema, config["destination_table"])
    last_watermark = None if is_full_load else max_value(target, watermark)
    if last_watermark is None:
        is_full_load = True

    increment = source if is_full_load else source.where(F.col(watermark) > F.lit(last_watermark))
    view_name = f"increment_{config['control_id']}"
    increment.createOrReplaceTempView(view_name)
    config["increment_view"] = view_name
    print(f"  {config['destination_table']}: " + ("full load" if is_full_load else f"{watermark} > {last_watermark}"))
    return increment, is_full_load


def overwrite_through_staging(df, layer, business_schema, table_name):
    """Parquet test mode only. A parquet folder cannot be overwritten while it is being read,
    so the new state is written next to it and the folders are swapped."""
    path = table_path(layer, business_schema, table_name)
    staging = path.with_name(f"_staging_{path.name}")
    shutil.rmtree(staging, ignore_errors=True)
    df.write.mode("overwrite").parquet(staging.as_posix())
    shutil.rmtree(path)
    staging.rename(path)
    reregister_table(layer, business_schema, table_name)


def merge_table(df, layer, business_schema, table_name, key_columns):
    """Upsert: rows whose key exists are updated, new keys are inserted."""
    keys = [column.strip() for column in key_columns.split(",")]
    target = table_full_name(layer, business_schema, table_name)
    rows = df.count()
    if rows == 0:
        return 0
    if TABLE_FORMAT == "delta":
        view_name = f"merge_source_{uuid.uuid4().hex}"
        df.createOrReplaceTempView(view_name)
        match_condition = " AND ".join(f"target.{key} = source.{key}" for key in keys)
        spark.sql(f"""
            MERGE INTO {target} AS target
            USING {view_name} AS source
            ON {match_condition}
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
        spark.catalog.dropTempView(view_name)
        spark.catalog.refreshTable(target)
    else:
        untouched = read_fresh(target).join(df.select(*keys), on=keys, how="left_anti")
        overwrite_through_staging(untouched.unionByName(df), layer, business_schema, table_name)
    return rows


def changed_partitions(config, increment):
    """Predicate covering the target periods that the new rows touch, or None if nothing is new."""
    source_expression = config["partition_source_column"]
    bounds = increment.selectExpr(
        f"MIN({source_expression}) AS low",
        f"MAX({source_expression}) AS high",
    ).collect()[0]
    if bounds["low"] is None:
        return None
    column = config["partition_column"]
    return f"{column} BETWEEN {sql_literal(bounds['low'])} AND {sql_literal(bounds['high'])}"


def replace_partitions(df, layer, business_schema, table_name, predicate):
    """Rebuild only the rows matching predicate; the rest of the table is untouched."""
    target = table_full_name(layer, business_schema, table_name)
    df = df.where(predicate)
    rows = df.count()
    if TABLE_FORMAT == "delta":
        (
            df.write.format("delta")
            .mode("overwrite")
            .option("replaceWhere", predicate)
            .save(table_path(layer, business_schema, table_name).as_posix())
        )
        spark.catalog.refreshTable(target)
    else:
        untouched = read_fresh(target).where(f"NOT coalesce({predicate}, false)")
        overwrite_through_staging(untouched.unionByName(df), layer, business_schema, table_name)
    return rows

## Slowly changing dimension, type 2 (Silver)
Keeps every version of a member. Each version has `ValidFrom` (inclusive), `ValidTo`
(exclusive, 9999-12-31 while current) and `IsCurrent`. The flow's script returns the latest
snapshot; comparing it with the current versions is the change capture:

| Change | What happens |
|---|---|
| New member | First version, valid from 1900-01-01, so its full history in the facts finds it |
| Changed member (row hash differs) | Current version closed today, new version opened today. A second change on the same day replaces that day's version |
| Member missing from the snapshot | Current version closed today (a delete), no new version |
| Unchanged | Nothing |

The surrogate key the facts join on is added in Gold (a hash of the business key and
`ValidFrom`, so it is the same on every rebuild). A full refresh never rebuilds this table:
history cannot be recreated from one snapshot. Bronze keeps every snapshot, so it could be
replayed if ever needed.

In [ ]:
from pyspark.sql.window import Window

FIRST_VALID_FROM = "1900-01-01"
OPEN_VALID_TO = "9999-12-31"


def apply_scd2(df, layer, business_schema, table_name, key_columns, batch_id):
    keys = [column.strip() for column in key_columns.split(",")]
    attributes = [column for column in df.columns if column not in keys]
    version_columns = [*df.columns, "ValidFrom", "ValidTo", "IsCurrent", "_row_hash", "_batch_id"]
    version_key = [*keys, "ValidFrom"]
    source = df.withColumn("_row_hash", F.sha2(F.to_json(F.struct(*attributes)), 256))

    if not table_exists(layer, business_schema, table_name):
        first_versions = (
            source
            .withColumn("ValidFrom", F.to_date(F.lit(FIRST_VALID_FROM)))
            .withColumn("ValidTo", F.to_date(F.lit(OPEN_VALID_TO)))
            .withColumn("IsCurrent", F.lit(True))
            .withColumn("_batch_id", F.lit(batch_id))
        )
        return write_table(first_versions.select(version_columns), layer, business_schema, table_name)

    target = table_full_name(layer, business_schema, table_name)
    history = read_fresh(target)
    current = history.where("IsCurrent")
    change_date = F.current_date()

    compared = source.join(
        current.select(
            *keys,
            F.col("_row_hash").alias("_current_hash"),
            F.col("ValidFrom").alias("_current_from"),
        ),
        on=keys,
        how="left",
    )
    new_members = compared.where("_current_hash IS NULL").withColumn("ValidFrom", F.to_date(F.lit(FIRST_VALID_FROM)))
    changed = compared.where("_current_hash IS NOT NULL AND _current_hash <> _row_hash")
    changed_earlier = changed.where(F.col("_current_from") < change_date)
    changed_today = changed.where(F.col("_current_from") >= change_date)
    removed = current.join(source.select(*keys), on=keys, how="left_anti")

    def open_version(rows):
        return (
            rows
            .withColumn("ValidTo", F.to_date(F.lit(OPEN_VALID_TO)))
            .withColumn("IsCurrent", F.lit(True))
            .withColumn("_batch_id", F.lit(batch_id))
            .select(version_columns)
        )

    inserts = open_version(
        new_members.unionByName(changed_earlier.withColumn("ValidFrom", change_date))
    )
    replacements = open_version(changed_today.withColumn("ValidFrom", F.col("_current_from")))
    closures = (
        current.join(changed_earlier.select(*keys).unionByName(removed.select(*keys)), on=keys)
        .withColumn("ValidTo", change_date)
        .withColumn("IsCurrent", F.lit(False))
        .withColumn("_batch_id", F.lit(batch_id))
        .select(version_columns)
    )

    counts = {
        "new": new_members.count(),
        "changed": changed.count(),
        "removed": removed.count(),
    }
    print(f"  {table_name}: {counts['new']} new, {counts['changed']} changed, {counts['removed']} removed")
    if sum(counts.values()) == 0:
        return 0

    if TABLE_FORMAT == "delta":
        staged = (
            inserts.withColumn("_action", F.lit("insert"))
            .unionByName(replacements.withColumn("_action", F.lit("replace")))
            .unionByName(closures.withColumn("_action", F.lit("close")))
        )
        view_name = f"scd2_source_{uuid.uuid4().hex}"
        staged.createOrReplaceTempView(view_name)
        match_condition = " AND ".join(f"target.{column} = source.{column}" for column in version_key)
        insert_columns = ", ".join(version_columns)
        insert_values = ", ".join(f"source.{column}" for column in version_columns)
        replace_set = ", ".join(f"target.{column} = source.{column}" for column in version_columns)
        spark.sql(f"""
            MERGE INTO {target} AS target
            USING {view_name} AS source
            ON {match_condition}
            WHEN MATCHED AND source._action = 'close' THEN UPDATE SET
                target.ValidTo = source.ValidTo,
                target.IsCurrent = false,
                target._batch_id = source._batch_id
            WHEN MATCHED AND source._action = 'replace' THEN UPDATE SET {replace_set}
            WHEN NOT MATCHED AND source._action = 'insert' THEN INSERT ({insert_columns})
                VALUES ({insert_values})
        """)
        spark.catalog.dropTempView(view_name)
        spark.catalog.refreshTable(target)
    else:
        touched = closures.select(*version_key).unionByName(replacements.select(*version_key))
        untouched = history.join(touched, on=version_key, how="left_anti")
        overwrite_through_staging(
            untouched.select(version_columns).unionByName(closures).unionByName(replacements).unionByName(inserts),
            layer, business_schema, table_name,
        )
    return sum(counts.values())

## Scripts
Silver and Gold scripts are one plain `SELECT` each. `{{bronze}}`, `{{silver}}` and `{{gold}}`
resolve to the layer databases for the flow's business schema. An incremental script reads
`{{increment}}`: the new rows of its source (see Incremental loads).

In [ ]:
SCRIPT_FOLDERS = {
    "SILVER": "03_silver/030_scripts",
    "GOLD": "04_gold/040_scripts",
}


def read_script(config):
    script_path = PIPELINE_ROOT / SCRIPT_FOLDERS[config["layer"]] / config["source_object"]
    script_sql = script_path.read_text(encoding="utf-8")
    schema = config["destination_schema"]
    for layer in LAYERS:
        script_sql = script_sql.replace("{{" + layer + "}}", layer_database(layer, schema))
    if "{{increment}}" in script_sql:
        script_sql = script_sql.replace("{{increment}}", config["increment_view"])
    return script_sql


def run_script(config):
    return spark.sql(read_script(config))

## Running one flow
Looks up the flow, logs the start, runs its loader, logs the result. Retries up to the flow's
`retry_count`. A loader only does the work and returns the number of rows written.

In [ ]:
def execute_flow(control_id, batch_id, loader, triggered_by="engine", full_refresh=False):
    config = get_flow_config(control_id)
    config["full_refresh"] = full_refresh
    attempts = max(1, int(config["retry_count"] or 1))
    last_error = None

    for attempt in range(1, attempts + 1):
        run_id = str(uuid.uuid4())
        started = time.time()
        common = dict(
            run_id=run_id,
            control_id=control_id,
            batch_id=batch_id,
            pipeline_name=config["pipeline_name"],
            layer=config["layer"],
            retry_attempt=attempt,
            source_object=config["source_object"],
            destination_table=config["destination_table"],
            triggered_by=triggered_by,
        )
        append_run_event(status="RUNNING", **common)
        try:
            rows_written = loader(config, batch_id)
            append_run_event(
                status="SUCCEEDED",
                duration_seconds=round(time.time() - started, 1),
                rows_written=rows_written,
                **common,
            )
            return "SUCCEEDED"
        except QualityCheckFailed as error:
            # A failed check is a data problem, not a glitch: retrying will not fix it.
            append_run_event(
                status="FAILED",
                duration_seconds=round(time.time() - started, 1),
                error_message=str(error),
                **common,
            )
            return f"FAILED: {error}"
        except Exception as error:
            last_error = error
            append_run_event(
                status="FAILED",
                duration_seconds=round(time.time() - started, 1),
                error_message=f"{error}\n{traceback.format_exc()}"[:4000],
                **common,
            )
            print(f"  attempt {attempt} of {attempts} failed for {config['pipeline_name']}: {error}")

    return f"FAILED: {last_error}"


class QualityCheckFailed(Exception):
    pass

## Dependency waves
Same engine as the Databricks Silver orchestrator: flows whose dependencies are met run in
parallel within a wave, the next wave waits, and anything whose dependency failed is skipped.

A dependency outside the layer being run (for example a Gold table that waits for a Silver
quality check) counts as met only if that flow's latest run succeeded.

In [ ]:
LOADER_FOLDERS = {
    "BRONZE": "02_bronze/020_ingestion",
    "SILVER": "03_silver/030_loaders",
    "GOLD": "04_gold/040_loaders",
}

LOADERS = globals().get("LOADERS", {})


def run_layer(layer, batch_id=None, max_workers=4, full_refresh=False):
    layer = layer.upper()
    batch_id = batch_id or new_batch_id()
    registry = get_registry(layer)
    for notebook_name in sorted(set(registry.values())):
        run_notebook(f"{LOADER_FOLDERS[layer]}/{notebook_name}.ipynb")

    flows = get_active_flows(layer)
    flows_by_id = {flow["control_id"]: flow for flow in flows}
    print(f"{layer}: {len(flows)} active flow(s), batch {batch_id}")

    # ---- build waves
    remaining = set(flows_by_id)
    placed = set()
    waves = []
    while remaining:
        wave = sorted(
            cid for cid in remaining
            if all(dep not in flows_by_id or dep in placed for dep in dependencies_of(flows_by_id[cid]))
        )
        if not wave:
            print("WARNING: circular or missing dependency, these flows will NOT run:", sorted(remaining))
            break
        waves.append(wave)
        placed.update(wave)
        remaining -= set(wave)

    # ---- run waves
    results = {}
    for wave_number, wave in enumerate(waves):
        jobs = []
        for cid in wave:
            flow = flows_by_id[cid]
            blocked_by = [
                dep for dep in dependencies_of(flow)
                if (dep in flows_by_id and results.get(dep) != "SUCCEEDED")
                or (dep not in flows_by_id and latest_status(dep) != "SUCCEEDED")
            ]
            notebook_name = registry.get(flow["source_type"])
            if blocked_by:
                results[cid] = f"SKIPPED: dependency {blocked_by} did not succeed"
            elif not notebook_name:
                results[cid] = f"SKIPPED: no loader registered for source_type {flow['source_type']}"
            else:
                jobs.append((cid, LOADERS[notebook_name]))
                continue
            append_run_event(
                run_id=str(uuid.uuid4()),
                control_id=cid,
                batch_id=batch_id,
                pipeline_name=flow["pipeline_name"],
                layer=layer,
                status="SKIPPED",
                retry_attempt=0,
                source_object=flow["source_object"],
                destination_table=flow["destination_table"],
                error_message=results[cid],
                triggered_by="engine",
            )

        print(f"  wave {wave_number}: {len(jobs)} flow(s)")
        with ThreadPoolExecutor(max_workers=max_workers) as pool:
            futures = {
                pool.submit(execute_flow, cid, batch_id, loader, "engine", full_refresh): cid
                for cid, loader in jobs
            }
            for future in as_completed(futures):
                results[futures[future]] = future.result()

    # ---- summary
    for cid in sorted(results):
        print(f"  {cid:>4}  {flows_by_id[cid]['pipeline_name']:<38} {results[cid][:110]}")
    failed = {cid: result for cid, result in results.items() if result != "SUCCEEDED"}
    print(f"{layer}: {len(results) - len(failed)} succeeded, {len(failed)} failed or skipped")
    return results